In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

d:\Machine Learing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
loader = TextLoader("../data/docs.txt")
documents = loader.load()
print(documents)

[Document(metadata={'source': '../data/docs.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a technique that enhances large language models\nby retrieving relevant external documents and providing them as context during response generation.\n\nRAG reduces hallucinations and allows LLMs to answer questions based on private or updated data.\n\nVector databases store document embeddings and enable fast similarity search.')]


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)
docs = text_splitter.split_documents(documents)

In [5]:
embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

In [6]:
vectorstore = Chroma.from_documents(
    docs,
    embedding,
    persist_directory="./chroma_db"
)
vectorstore.persist()
print("documeent ingested successgully")

documeent ingested successgully


C:\Users\Acer\AppData\Local\Temp\ipykernel_16020\1699588868.py:6: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


Rag query pipeline

In [7]:
from langchain_community.llms import Ollama
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [8]:
#load embeddings
embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

In [9]:
#load vector db
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embedding
)

C:\Users\Acer\AppData\Local\Temp\ipykernel_16020\2655942119.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [10]:
#creating retrrever
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [11]:
#LLM with llama3
llm = Ollama(model='llama3')

C:\Users\Acer\AppData\Local\Temp\ipykernel_16020\1519148171.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model='llama3')


In [12]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True
)

In [15]:
query = "What is rag and why it is useful"
response = qa_chain.invoke({"query":query})

In [16]:
print(response["result"])

According to the given context, RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by retrieving relevant external documents and providing them as context during response generation. This makes RAG useful because it reduces hallucinations and allows LLMs to answer questions based on private or updated data.
